In [5]:
import urllib.request
import os

os.makedirs("data", exist_ok=True)

for month in ["01", "02", "03"]:
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{month}.parquet"
    output = f"data/yellow_tripdata_2024-{month}.parquet"

    print(f"Downloading 2024-{month}...")
    urllib.request.urlretrieve(url, output)

print("Download complete!")

In [6]:
# ! pip install pyarrow duckdb

In [1]:
import os
import time
import pyarrow as pa
import pyarrow.csv as pcsv
import pyarrow.parquet as pq
import duckdb
import pandas as pd

## 1. Explore the Dataset

Let's check the file sizes first

In [2]:
for f in os.listdir("data"):
    size = os.path.getsize(f"data/{f}") / (1024*1024)
    print(f"{f}: {size:.2f} MB")

yellow_tripdata_2024-01.parquet: 47.65 MB
yellow_tripdata_2024-02.parquet: 48.02 MB
yellow_tripdata_2024-03.parquet: 57.30 MB


- How many rows does each monthly file have? How many in total?


In [3]:
data_files = [
    "data/yellow_tripdata_2024-01.parquet",
    "data/yellow_tripdata_2024-02.parquet",
    "data/yellow_tripdata_2024-03.parquet"
]

In [4]:
rows_per_file = {}

for file in data_files:
    parquet_file = pq.ParquetFile(file)
    rows = parquet_file.metadata.num_rows
    rows_per_file[file] = rows
    print(f"{file}: {rows:,} rows")

total_rows = sum(rows_per_file.values())

print("\nTotal rows:", f"{total_rows:,}")

data/yellow_tripdata_2024-01.parquet: 2,964,624 rows
data/yellow_tripdata_2024-02.parquet: 3,007,526 rows
data/yellow_tripdata_2024-03.parquet: 3,582,628 rows

Total rows: 9,554,778


- What columns are available? What are their data types?

In [5]:
table = pq.read_table(data_files[0])

print("Columns and Data Types:\n")
print(table.schema)

Columns and Data Types:

VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double


In [6]:
sample_df = table.slice(0, 5).to_pandas()
sample_df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [7]:
print("Column names:\n")
for col in table.schema.names:
    print(col)

Column names:

VendorID
tpep_pickup_datetime
tpep_dropoff_datetime
passenger_count
trip_distance
RatecodeID
store_and_fwd_flag
PULocationID
DOLocationID
payment_type
fare_amount
extra
mta_tax
tip_amount
tolls_amount
improvement_surcharge
total_amount
congestion_surcharge
Airport_fee


## 2. MongoDB — Loading and Querying Semi-structured Data

### 2.1. Load data into MongoDB

In [8]:
os.makedirs("exports/json", exist_ok=True)

tables = [pq.read_table(f) for f in data_files]
combined_table = pa.concat_tables(tables)

print("Total rows in combined table:", combined_table.num_rows)
print("Total columns:", combined_table.num_columns)

Total rows in combined table: 9554778
Total columns: 19


In [9]:
sample_size = 100_000
sample_table = combined_table.slice(0, sample_size)

print("Sample rows:", sample_table.num_rows)

Sample rows: 100000


In [10]:
sample_df = sample_table.to_pandas()
json_path = "exports/json/trips_sample.json"

sample_df.to_json(json_path, orient="records", lines=True)

print(f"JSON Lines sample exported to: {json_path}")
print(f"File size: {os.path.getsize(json_path) / (1024 * 1024):.2f} MB")

JSON Lines sample exported to: exports/json/trips_sample.json
File size: 37.62 MB


### 2.2 Explore and query

#### 1. How many documents are in the collection?

```javascript
db.trips.countDocuments()
````

---

#### 2. Find all trips with more than 4 passengers

```javascript
db.trips.find({ passenger_count: { $gt: 4 } }).limit(10)
```

---

#### 3. Find the average `tip_amount` grouped by `payment_type`

```javascript
db.trips.aggregate([
  {
    $group: {
      _id: "$payment_type",
      avg_tip_amount: { $avg: "$tip_amount" }
    }
  },
  {
    $sort: { _id: 1 }
  }
])
```

---

#### 4. Find the top 5 longest trips by `trip_distance`

```javascript
db.trips.find(
  {},
  {
    trip_distance: 1,
    passenger_count: 1,
    fare_amount: 1,
    tip_amount: 1,
    payment_type: 1
  }
).sort({ trip_distance: -1 }).limit(5)
```

```

## 3 Format Comparison

### 3.1 Read and combine the data

In [11]:
import os

os.makedirs("exports/csv", exist_ok=True)
os.makedirs("exports/json", exist_ok=True)
os.makedirs("exports/parquet_compressed", exist_ok=True)

In [12]:
print("Combined rows:", combined_table.num_rows)
print("Combined columns:", combined_table.num_columns)

Combined rows: 9554778
Combined columns: 19


### 3.2 Export to other formats

In [13]:
csv_path = "exports/csv/yellow_tripdata_2024_Q1.csv"

start = time.perf_counter()
combined_df = combined_table.to_pandas()
combined_df.to_csv(csv_path, index=False)
csv_write_time = time.perf_counter() - start

print(f"CSV exported to: {csv_path}")
print(f"CSV write time: {csv_write_time:.2f} seconds")

CSV exported to: exports/csv/yellow_tripdata_2024_Q1.csv
CSV write time: 223.39 seconds


In [14]:
json_sample_path = "exports/json/trips_sample_100k.json"
sample_size = 100_000

json_sample_table = combined_table.slice(0, sample_size)
json_sample_df = json_sample_table.to_pandas()

start = time.perf_counter()
json_sample_df.to_json(json_sample_path, orient="records", lines=True)
json_write_time = time.perf_counter() - start

print(f"JSON Lines sample exported to: {json_sample_path}")
print(f"JSON write time: {json_write_time:.2f} seconds")

JSON Lines sample exported to: exports/json/trips_sample_100k.json
JSON write time: 1.15 seconds


### 3.3 Compare file sizes

In [15]:
def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

In [16]:
parquet_total_size_mb = sum(file_size_mb(file) for file in data_files)
csv_size_mb = file_size_mb(csv_path)
json_sample_size_mb = file_size_mb(json_sample_path)

total_rows = combined_table.num_rows
json_extrapolated_size_mb = json_sample_size_mb * (total_rows / sample_size)

size_comparison = pd.DataFrame([
    {
        "Format": "Parquet original files",
        "File Size (MB)": parquet_total_size_mb,
        "Ratio vs Parquet": 1.0
    },
    {
        "Format": "CSV",
        "File Size (MB)": csv_size_mb,
        "Ratio vs Parquet": csv_size_mb / parquet_total_size_mb
    },
    {
        "Format": "JSON Lines (100K rows, extrapolated total)",
        "File Size (MB)": json_extrapolated_size_mb,
        "Ratio vs Parquet": json_extrapolated_size_mb / parquet_total_size_mb
    }
])

size_comparison

,Format,File Size (MB),Ratio vs Parquet
0,Parquet original files,152.959065,1.000000
1,CSV,960.289380,6.278081
2,"JSON Lines (100K rows, extrapolated total)",3594.073279,23.496962


### 3.4 Compare read times

In [17]:
def time_parquet_read(files):
    start = time.perf_counter()
    tables = [pq.read_table(file) for file in files]
    _ = pa.concat_tables(tables)
    return time.perf_counter() - start

In [18]:
def time_csv_read(path):
    start = time.perf_counter()
    _ = pd.read_csv(path)
    return time.perf_counter() - start

In [19]:
parquet_read_time = time_parquet_read(data_files)
csv_read_time = time_csv_read(csv_path)

read_comparison = pd.DataFrame([
    {"Format": "Parquet", "Read Time (s)": parquet_read_time},
    {"Format": "CSV", "Read Time (s)": csv_read_time}
])

read_comparison

C:\Users\Sami Malik\AppData\Local\Temp\ipykernel_24252\613176342.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  _ = pd.read_csv(path)


,Format,Read Time (s)
0,Parquet,1.185699
1,CSV,72.310105


### 3.5 Compression comparison

In [20]:
compression_results = []
compressions = ["snappy", "gzip", "zstd", None]

In [21]:
for compression in compressions:
    label = compression if compression is not None else "none"
    output_path = f"exports/parquet_compressed/yellow_tripdata_2024_Q1_{label}.parquet"
    
    start_write = time.perf_counter()
    pq.write_table(combined_table, output_path, compression=compression)
    write_time = time.perf_counter() - start_write
    
    size_mb = file_size_mb(output_path)
    
    start_read = time.perf_counter()
    _ = pq.read_table(output_path)
    read_time = time.perf_counter() - start_read
    
    compression_results.append({
        "Compression": label,
        "File Size (MB)": size_mb,
        "Write Time (s)": write_time,
        "Read Time (s)": read_time
    })

compression_df = pd.DataFrame(compression_results)
compression_df.sort_values("File Size (MB)")

,Compression,File Size (MB),Write Time (s),Read Time (s)
1,gzip,142.944144,117.231433,1.293802
2,zstd,151.070868,9.508724,0.847797
0,snappy,186.357588,28.463713,0.973575
3,none,238.291112,8.082028,0.702267


#### Interpretation

Parquet was significantly more storage-efficient than CSV and JSON Lines.  
CSV was much larger because it stores everything as plain text and repeats column values row by row.  
JSON Lines is even less efficient for this tabular dataset because every row repeats the field names.

For compression, `gzip` and `zstd` usually produce smaller files than `snappy`, but they may take longer to write or read.  
`snappy` is often a strong choice for frequently queried data because it balances compression and speed well.

But in our case, we can choose `zstd`because it offers low write and read time compared to others. And if storage is not a problem, we can continue without any compression in this case.

### 3.6 Column pruning

#### Column Pruning Experiment

Analytical question:
What is the average tip amount per payment type?

Required columns:
- payment_type
- tip_amount

In [69]:
start = time.perf_counter()
full_table = pq.read_table(data_files)
full_read_time = time.perf_counter() - start

full_df = full_table.to_pandas()
full_result = (
    full_df.groupby("payment_type", dropna=False)["tip_amount"]
    .mean()
    .reset_index(name="avg_tip_amount")
    .sort_values("payment_type")
)


print(f"Read + query time using ALL columns: {full_read_time:.4f} seconds")
full_result

Read + query time using ALL columns: 1.1761 seconds


,payment_type,avg_tip_amount
0,0,0.916933
1,1,4.209618
2,2,0.002198
3,3,0.012355
4,4,0.047065


In [70]:
needed_columns = ["payment_type", "tip_amount"]

start = time.perf_counter()
pruned_table = pq.read_table(data_files, columns=needed_columns)
pruned_read_time = time.perf_counter() - start

pruned_df = pruned_table.to_pandas()
pruned_result = (
    pruned_df.groupby("payment_type", dropna=False)["tip_amount"]
    .mean()
    .reset_index(name="avg_tip_amount")
    .sort_values("payment_type")
)

print(f"Read + query time using ONLY needed columns: {pruned_read_time:.4f} seconds")
pruned_result

Read + query time using ONLY needed columns: 0.2564 seconds


,payment_type,avg_tip_amount
0,0,0.916933
1,1,4.209618
2,2,0.002198
3,3,0.012355
4,4,0.047065


In [71]:
column_pruning_comparison = pd.DataFrame([
    {
        "Method": "Read all columns",
        "Time (s)": full_read_time
    },
    {
        "Method": "Read only needed columns",
        "Time (s)": pruned_read_time
    }
])

column_pruning_comparison["Speedup"] = (
    column_pruning_comparison["Time (s)"].max() / column_pruning_comparison["Time (s)"]
)

column_pruning_comparison

,Method,Time (s),Speedup
0,Read all columns,1.176145,1.000000
1,Read only needed columns,0.256351,4.588029


In [32]:
improvement_percent = ((full_read_time - pruned_read_time) / full_read_time) * 100
print(f"Column pruning improved performance by {improvement_percent:.2f}%")

Column pruning improved performance by 76.39%


#### Interpretation

Column pruning was faster because the analytical query only needed `payment_type` and `tip_amount`.
When reading the full dataset, Parquet had to load many unnecessary columns.
When reading only the required columns, less data was read from disk and converted into memory.

This shows one of the main advantages of Parquet for analytical workloads:
query engines can avoid reading irrelevant columns, which reduces I/O and improves performance.

## 4  Partitioning Strategy

### 4.1 Choose a partition column

In [22]:
partition_output_month = "exports/partitioned_by_month"
os.makedirs(partition_output_month, exist_ok=True)

In [23]:
combined_table.schema

VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double

In [24]:
import pyarrow.compute as pc

# extract the month from tpep_pickup_datetime column
months = pc.month(combined_table.column("tpep_pickup_datetime"))

# add a new column to a table
table_with_month = combined_table.append_column("pickup_month", months)

# write the partitioned dataset
pq.write_to_dataset(
    table_with_month,
    partition_output_month,
    partition_cols=["pickup_month"]
)


In [25]:
table_with_month.schema

VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
pickup_month: int64

### 4.2 Inspect the partitioned output

In [26]:
month_partitions = sorted(
    [d for d in os.listdir(partition_output_month) if os.path.isdir(os.path.join(partition_output_month, d))]
)

month_partitions

['pickup_month=1',
 'pickup_month=12',
 'pickup_month=2',
 'pickup_month=3',
 'pickup_month=4']

In [27]:
print("Number of partition directories:", len(month_partitions))

Number of partition directories: 5


In [28]:
def get_directory_size_mb(path):
    total_size = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

In [29]:
partition_sizes = []

for partition in month_partitions:
    path = os.path.join(partition_output_month, partition)
    partition_sizes.append({
        "Partition": partition,
        "Size (MB)": get_directory_size_mb(path)
    })

partition_sizes_df = pd.DataFrame(partition_sizes)
partition_sizes_df

,Partition,Size (MB)
0,pickup_month=1,119.107571
1,pickup_month=12,0.042051
2,pickup_month=2,119.616219
3,pickup_month=3,143.391085
4,pickup_month=4,0.011581


#### Partition inspection interpretation

Five partition directories were created: `pickup_month=1`, `pickup_month=2`, `pickup_month=3`, `pickup_month=4`, and `pickup_month=12`.

Most of the data is concentrated in the expected months (January, February, and March). These partitions contain approximately 2.9–3.6 million rows each.

Two additional partitions (`pickup_month=4` and `pickup_month=12`) contain only a very small number of rows (2 and 15 rows respectively), likely due to a few trips whose timestamps fall slightly outside the main months.

Overall, the main partitions (months 1–3) are reasonably balanced in terms of both row counts and file sizes, which indicates that partitioning by month is a suitable strategy for this dataset.

### 4.3 Partition pruning

In [36]:
import pyarrow.dataset as ds

single_parquet_path = "exports/parquet_compressed/yellow_tripdata_2024_Q1_zstd.parquet"

# Query January from the partitioned dataset
start = time.perf_counter()

partitioned_dataset = ds.dataset(partition_output_month, format="parquet", partitioning="hive")
jan_partitioned_table = partitioned_dataset.to_table(filter=ds.field("pickup_month") == 1)

partitioned_read_time = time.perf_counter() - start

print(f"Partitioned dataset read time (January only): {partitioned_read_time:.4f} seconds")
print("January rows from partitioned dataset:", jan_partitioned_table.num_rows)

Partitioned dataset read time (January only): 0.6749 seconds
January rows from partitioned dataset: 5929242


In [40]:
start = time.perf_counter()

full_single_table = pq.read_table(single_parquet_path)
jan_non_partitioned_table = full_single_table.filter(pc.equal(pc.month(full_single_table.column("tpep_pickup_datetime")), 1))
non_partitioned_read_time = time.perf_counter() - start

print(f"Non-partitioned file read + filter time (January): {non_partitioned_read_time:.4f} seconds")
print("January rows from non-partitioned file:", len(jan_non_partitioned_table))

Non-partitioned file read + filter time (January): 1.7127 seconds
January rows from non-partitioned file: 2964621


In [41]:
partition_pruning_comparison = pd.DataFrame([
    {
        "Method": "Partitioned dataset with pruning",
        "Time (s)": partitioned_read_time
    },
    {
        "Method": "Non-partitioned file + in-memory filter",
        "Time (s)": non_partitioned_read_time
    }
])

partition_pruning_comparison["Speedup"] = (
    partition_pruning_comparison["Time (s)"].max() / partition_pruning_comparison["Time (s)"]
)

partition_pruning_comparison

,Method,Time (s),Speedup
0,Partitioned dataset with pruning,0.674876,2.537768
1,Non-partitioned file + in-memory filter,1.712678,1.000000


#### Partition pruning interpretation

When querying the partitioned dataset, only the partition corresponding to January is read from disk. This significantly reduces the amount of data that needs to be scanned.

In contrast, when querying the non-partitioned Parquet file, the entire dataset must be read first before applying the filter to select January trips.

Because of this, the partitioned dataset typically results in faster query times and lower I/O usage. This demonstrates the advantage of partition pruning for analytical workloads where queries frequently filter on the partition column.

### 4.4 Over-partitioning

In [42]:
partition_output_day = "exports/partitioned_by_day"
os.makedirs(partition_output_day, exist_ok=True)

In [43]:
years = pc.year(combined_table.column("tpep_pickup_datetime"))
months = pc.month(combined_table.column("tpep_pickup_datetime"))
days = pc.day(combined_table.column("tpep_pickup_datetime"))

table_with_ymd = (
    combined_table
    .append_column("pickup_year", years)
    .append_column("pickup_month", months)
    .append_column("pickup_day", days)
)

table_with_ymd.schema

VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
pickup_year: int64
pickup_month: int64
pickup_day: int64

In [74]:
# Write the dataset partitioned by year / month / day
pq.write_to_dataset(
    table_with_ymd,
    root_path=partition_output_day,
    partition_cols=["pickup_year", "pickup_month", "pickup_day"]
)

In [44]:
day_partition_dirs = []

for root, dirs, files in os.walk(partition_output_day):
    if any(f.endswith(".parquet") for f in files):
        day_partition_dirs.append(root)

print("Number of leaf partition directories:", len(day_partition_dirs))

Number of leaf partition directories: 96


In [45]:
day_partition_sizes = []

for path in sorted(day_partition_dirs):
    relative_path = os.path.relpath(path, partition_output_day)
    day_partition_sizes.append({
        "Partition Path": relative_path,
        "Size (MB)": get_directory_size_mb(path)
    })

day_partition_sizes_df = pd.DataFrame(day_partition_sizes)
day_partition_sizes_df.head(10)

,Partition Path,Size (MB)
0,pickup_year=2002\pickup_month=12\pickup_day=31,0.013128
1,pickup_year=2008\pickup_month=12\pickup_day=31,0.005714
2,pickup_year=2009\pickup_month=1\pickup_day=1,0.016778
3,pickup_year=2023\pickup_month=12\pickup_day=31,0.006211
4,pickup_year=2024\pickup_month=1\pickup_day=1,1.768921
5,pickup_year=2024\pickup_month=1\pickup_day=10,1.919933
6,pickup_year=2024\pickup_month=1\pickup_day=11,2.093298
7,pickup_year=2024\pickup_month=1\pickup_day=12,2.096299
8,pickup_year=2024\pickup_month=1\pickup_day=13,2.081652
9,pickup_year=2024\pickup_month=1\pickup_day=14,1.949863


In [46]:
day_partition_sizes_df["Size (MB)"].describe()

count    96.000000
mean      2.019824
std       0.543560
min       0.005714
25%       1.947496
50%       2.094799
75%       2.270197
max       2.746801
Name: Size (MB), dtype: float64

In [47]:
pickup_year_arr = table_with_ymd.column("pickup_year").to_pylist()
pickup_month_arr = table_with_ymd.column("pickup_month").to_pylist()
pickup_day_arr = table_with_ymd.column("pickup_day").to_pylist()

In [48]:
from collections import Counter

day_counts = Counter(zip(pickup_year_arr, pickup_month_arr, pickup_day_arr))

day_count_rows = [
    {
        "pickup_year": y,
        "pickup_month": m,
        "pickup_day": d,
        "row_count": count
    }
    for (y, m, d), count in sorted(day_counts.items())
]

day_counts_df = pd.DataFrame(day_count_rows)
day_counts_df.head(10)

,pickup_year,pickup_month,pickup_day,row_count
0,2002,12,31,4
1,2008,12,31,1
2,2009,1,1,4
3,2023,12,31,10
4,2024,1,1,81013
5,2024,1,2,75519
6,2024,1,3,82427
7,2024,1,4,102901
8,2024,1,5,103178
9,2024,1,6,97117


#### Over-partitioning interpretation

Partitioning by year, month, and day created many more partition directories than partitioning only by month. Each partition became much smaller, and many partitions contain relatively little data.

This is generally not a good strategy for this dataset. Although day-level partitioning may help very selective queries, it also creates many small files and directories, which increases filesystem overhead and metadata management costs.

For this taxi dataset, month is a better partitioning column because it keeps partitions reasonably large and still allows effective pruning for common time-based queries.

### 4.5 Metadata inspection

In [49]:
metadata = pq.read_metadata(single_parquet_path)
print(metadata)

  created_by: parquet-cpp-arrow version 23.0.1
  num_columns: 19
  num_rows: 9554778
  num_row_groups: 10
  format_version: 2.6
  serialized_size: 25709


In [50]:
num_row_groups = metadata.num_row_groups
print("Number of row groups:", num_row_groups)

Number of row groups: 10


In [51]:
schema_names = metadata.schema.names
target_column = "trip_distance"
target_column_index = schema_names.index(target_column)

print(f"Column '{target_column}' index:", target_column_index)

Column 'trip_distance' index: 4


In [52]:
row_group_stats = []

for rg_idx in range(metadata.num_row_groups):
    row_group = metadata.row_group(rg_idx)
    column_chunk = row_group.column(target_column_index)
    stats = column_chunk.statistics

    row_group_stats.append({
        "row_group": rg_idx,
        "num_rows": row_group.num_rows,
        "min_value": stats.min if stats is not None else None,
        "max_value": stats.max if stats is not None else None,
        "null_count": stats.null_count if stats is not None else None
    })

row_group_stats_df = pd.DataFrame(row_group_stats)
row_group_stats_df

,row_group,num_rows,min_value,max_value,null_count
0,0,1048576,-0.0,10879.28,0
1,1,1048576,-0.0,15400.32,0
2,2,1048576,-0.0,312722.30,0
3,3,1048576,-0.0,98229.40,0
4,4,1048576,-0.0,63515.20,0
5,5,1048576,-0.0,222478.29,0
6,6,1048576,-0.0,66907.90,0
7,7,1048576,-0.0,61099.80,0
8,8,1048576,-0.0,176836.30,0
9,9,117594,-0.0,176744.79,0


In [53]:
row_group_stats_df.describe(include="all")

,row_group,num_rows,min_value,max_value,null_count
count,10.00000,1.000000e+01,10.0,10.000000,10.0
mean,4.50000,9.554778e+05,0.0,120481.358000,0.0
std,3.02765,2.944024e+05,0.0,98258.516657,0.0
min,0.00000,1.175940e+05,-0.0,10879.280000,0.0
25%,2.25000,1.048576e+06,0.0,61703.650000,0.0
50%,4.50000,1.048576e+06,-0.0,82568.650000,0.0
75%,6.75000,1.048576e+06,-0.0,176813.422500,0.0
max,9.00000,1.048576e+06,-0.0,312722.300000,0.0


#### Metadata interpretation

Each Parquet file is divided into row groups, and each row group stores statistics such as
minimum and maximum values for columns like `trip_distance`.

A query engine can use these statistics to skip row groups that cannot possibly match a filter.
For example, if a query asks for trips with `trip_distance > 50` and a row group has
a maximum `trip_distance` of 20, that row group can be skipped completely.

This reduces disk I/O and improves analytical query performance.